In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1. Create a mock banking dataset (1,000 applicants)
np.random.seed(42)
n_samples = 1000

data = pd.DataFrame({
    'Applicant_ID': range(1, n_samples + 1),
    'Income_EUR': np.random.randint(20000, 120000, n_samples),
    'Credit_Score': np.random.randint(300, 850, n_samples),
    'Age': np.random.randint(18, 75, n_samples)
})

# 2. Intentionally bake in an age bias into the historical loan decisions
# If the applicant is over 60, we drastically lower their approval chances on purpose
def historical_approval(row):
    if row['Age'] > 60:
        return 1 if (row['Credit_Score'] > 700 and np.random.rand() > 0.6) else 0
    else:
        return 1 if row['Credit_Score'] > 580 else 0

data['Loan_Approved'] = data.apply(historical_approval, axis=1)

# 3. Train the AI Model
X = data[['Income_EUR', 'Credit_Score', 'Age']]
y = data['Loan_Approved']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)

# 4. Calculate overall performance metrics
predictions = model.predict(X_test)
print(f"--- MODEL TRAINING COMPLETED ---")
print(f"Overall Model Accuracy: {accuracy_score(y_test, predictions) * 100:.2f}%\n")

# 5. Measure the Bias (The NIST AI RMF 'Measure' Step)
test_results = X_test.copy()
test_results['Actual'] = y_test
test_results['Predicted'] = predictions

young_apps = test_results[test_results['Age'] <= 60]
senior_apps = test_results[test_results['Age'] > 60]

young_approval_rate = (young_apps['Predicted'] == 1).mean()
senior_approval_rate = (senior_apps['Predicted'] == 1).mean()
disparate_impact_ratio = senior_approval_rate / young_approval_rate

print(f"--- ALGORITHMIC BIAS REPORT ---")
print(f"Approval Rate for Applicants <= 60: {young_approval_rate * 100:.2f}%")
print(f"Approval Rate for Applicants > 60: {senior_approval_rate * 100:.2f}%")
print(f"Disparate Impact Ratio: {disparate_impact_ratio:.2f}")
print("(A Disparate Impact Ratio below 0.80 means legal bias is present!)")

--- MODEL TRAINING COMPLETED ---
Overall Model Accuracy: 97.50%

--- ALGORITHMIC BIAS REPORT ---
Approval Rate for Applicants <= 60: 47.56%
Approval Rate for Applicants > 60: 0.00%
Disparate Impact Ratio: 0.00
(A Disparate Impact Ratio below 0.80 means legal bias is present!)
